In [11]:
import json

import pandas as pd
import yfinance as yf

from datetime import date

from utils.datetime import DATE_FORMAT
from utils.datapath import adj_share_counts_path, daily_prices_path, basket_500_path, basket_all_path
from utils.data import all_securities
from utils.market_data import get_stock_prices

In [2]:
start_date = "2014-01-01"
end_date = date.today().strftime(DATE_FORMAT)
stock_list = [security["code"] for security in all_securities]
black_list = ["2372"]

# prices = get_stock_prices(stock_list, start_date=start_date, end_date=end_date)
# prices_df = pd.DataFrame(prices).reset_index()
# prices_df.to_csv(daily_prices_path)


In [4]:
code = "2372"
ticker = yf.Ticker(f"{code}.T")
ticker.history()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2025-04-30 00:00:00+09:00,4.614366e+09,4.616021e+09,4.614366e+09,4.614366e+09,0,0.0,0.000000e+00
2025-05-01 00:00:00+09:00,4.614366e+09,4.617676e+09,4.614366e+09,4.616021e+09,0,0.0,0.000000e+00
2025-05-02 00:00:00+09:00,4.616021e+09,4.617676e+09,4.614366e+09,4.614366e+09,0,0.0,0.000000e+00
2025-05-07 00:00:00+09:00,4.614366e+09,4.619332e+09,4.614366e+09,4.616021e+09,0,0.0,0.000000e+00
2025-05-08 00:00:00+09:00,4.616021e+09,4.619332e+09,4.616021e+09,4.617676e+09,0,0.0,0.000000e+00
2025-05-09 00:00:00+09:00,4.617676e+09,4.622642e+09,4.617676e+09,4.617676e+09,0,0.0,0.000000e+00
2025-05-12 00:00:00+09:00,NaN,NaN,NaN,NaN,0,0.0,6.041566e-07


In [5]:
shares_df = pd.read_csv(adj_share_counts_path, index_col="Quarter")
prices_df = pd.read_csv(daily_prices_path)

shares_df = shares_df.drop(["2025 Q2"], axis=0)
prices_df = prices_df.drop(columns=["Unnamed: 0"] + black_list)

# shares_df

In [6]:
prices_df["Quarter"] = prices_df["Date"].apply(lambda date: f"{pd.Timestamp(date).year} Q{pd.Timestamp(date).quarter}")
quarterly_prices_df = prices_df.ffill().drop_duplicates(subset="Quarter", keep="last").drop(columns="Date").set_index("Quarter")
quarterly_prices_df = quarterly_prices_df.drop(["2025 Q2"], axis=0)

In [7]:
quarterly_prices_df

,6178,8316,8411,8306,9501,9432,8421,6758,4612,7203,...,4894,9330,5889,151A,135A,143A,9345,3045,5591,3350
Quarter,,,,,,,,,,,,,,,,,,,,,
2014 Q1,NaN,1118.873291,1312.161255,390.499817,416.000000,39.968506,143719.937500,386.126160,279.278168,849.510925,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,420.471039,NaN,330.000000
2014 Q2,NaN,1077.001221,1337.889893,427.690308,422.000000,44.940552,149962.015625,329.342896,382.846802,887.130737,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,454.613373,NaN,340.000000
2014 Q3,NaN,1149.833252,1282.166870,433.233612,384.000000,49.162498,158411.656250,389.846466,441.979553,953.309814,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,460.629761,NaN,580.000000
2014 Q4,NaN,1125.144287,1325.364136,464.178955,492.000000,44.765911,164273.109375,484.126221,631.143188,1114.825439,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,464.789795,NaN,510.000000
2015 Q1,NaN,1183.394531,1406.909668,525.686890,455.000000,53.957401,182454.578125,624.615906,791.074280,1254.741943,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,506.496735,NaN,470.000000
2015 Q2,NaN,1403.922974,1766.134644,621.889526,667.000000,64.680321,171165.609375,677.776733,621.173035,1227.800171,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,569.761108,NaN,470.000000
2015 Q3,NaN,1179.113159,1509.621948,512.526184,797.000000,61.662884,168114.515625,569.372131,376.756104,1058.104248,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,505.088409,NaN,400.000000
2015 Q4,1271.122437,1203.678223,1649.138550,541.643677,699.000000,71.374306,172004.640625,589.703247,535.533325,1136.577881,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,495.770905,NaN,300.000000
2016 Q1,1040.606201,891.652222,1163.426147,379.437256,619.000000,72.407448,175273.468750,568.291626,455.575684,919.690613,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,503.880615,NaN,310.000000


In [8]:
# quarterly_prices_df

In [9]:
market_cap_df = quarterly_prices_df.mul(shares_df)

In [10]:
jp_all = market_cap_df.apply(
    lambda x: x.sort_values(ascending=False).index.tolist(),
    axis=1
).to_frame(name='jp_all')

jp_500 = market_cap_df.apply(
    lambda x: x.sort_values(ascending=False).head(500).index.tolist(),
    axis=1
).to_frame(name='jp_500')

In [13]:
with open(basket_all_path, 'w', encoding='utf-8') as f:
    json.dump(jp_all["jp_all"].to_dict(), f)

with open(basket_500_path, 'w', encoding='utf-8') as f:
    json.dump(jp_500["jp_500"].to_dict(), f)

In [60]:
# TODO:
# 1. Convert dates in prices_df to quarters
# 2. For each quarter, for each stock, multiply quarterly price with shares to get market cap
# 3. For each quarter, get 500 stocks with highest market cap
# 4. Voila, you have your constituents


In [1]:
from utils.data import jp_500, jp_all

In [4]:
# jp_all

In [ ]:
# jp_500

{'2014 Q1': ['9031',
  '4651',
  '1301',
  '1332',
  '135A',
  '1375',
  '1376',
  '1377',
  '1379',
  '137A',
  '1381',
  '1382',
  '1383',
  '1384',
  '138A',
  '1401',
  '1407',
  '1414',
  '1417',
  '1419',
  '141A',
  '1420',
  '1429',
  '142A',
  '1430',
  '1431',
  '1433',
  '1434',
  '1435',
  '143A',
  '1443',
  '1444',
  '1446',
  '1447',
  '1449',
  '1451',
  '148A',
  '1491',
  '149A',
  '1514',
  '1515',
  '1518',
  '151A',
  '153A',
  '156A',
  '1605',
  '165A',
  '1662',
  '1663',
  '166A',
  '167A',
  '168A',
  '1711',
  '1712',
  '1716',
  '1717',
  '1718',
  '1719',
  '1720',
  '1721',
  '1723',
  '1724',
  '1726',
  '1730',
  '1736',
  '1738',
  '1743',
  '1757',
  '1758',
  '1762',
  '1764',
  '1766',
  '1768',
  '176A',
  '1770',
  '1771',
  '1775',
  '1776',
  '1777',
  '177A',
  '1780',
  '1783',
  '1786',
  '1787',
  '1788',
  '1789',
  '1793',
  '1795',
  '1798',
  '1799',
  '1801',
  '1802',
  '1803',
  '1805',
  '1807',
  '1808',
  '1810',
  '1811',
  '1812',